# Many-Batch Alignment from scratch

This notebook rebuilds the Many-Batch Alignment (MBA) model, one idea at a time, on a toy world small enough to print. By the end you will have written every piece of `src/multibatch.py` yourself, and Section 18 checks that your version gives the same warps as the real code to about 10⁻¹³ bins.

**How each section works.** Intuition in plain words → a tiny example done by hand → the maths, with the paper's equation number → code from scratch → a plot → an assertion that checks the result → an exercise with a hidden solution.

**Where to look things up.** `tutorial/CODE_MAP.md` maps every paper concept to the file, function and variable that implements it. The notation table below is used throughout, and there is a glossary at the end.

**Units.** Retention time is measured in *bins*. The real data has 200 bins over a 45-minute run, so 1 bin = 0.225 min. The toy uses the same grid, for a reason explained in Section 1.

## Notation

| Symbol | Meaning | Name in the code (`src/multibatch.py`) |
|---|---|---|
| T | number of retention-time bins (200) | `N_BINS` (from `gcmsalign.config`) |
| t | a retention time, in bins | `t` |
| K | number of batches | `n_batches`, `K` |
| k, l | batch indices | `k`, `l` |
| f_k(t) = t + d_k(t) | warp of batch k into the shared clock | `JointWarp.latent(k, t)` |
| d_k(t) | drift of batch k | `drift[k]` (values at the knots) |
| Q | number of knots | `n_knots`, `M` |
| u_0 … u_{Q−1} | knot positions (equally spaced) | `knots` |
| Δu | spacing between knots | `dk` |
| θ_k = (c_k, δ_k1 … δ_k,Q−1) | parameters of batch k: first knot value + increments | `theta[k*M:(k+1)*M]` |
| L | cumulative-sum matrix, d_k(u) = Lθ_k | `Lcum` |
| B(t) | hat-basis (linear-interpolation) weights | `_hat_basis(t, knots)` |
| (a_j, b_j) | anchor j: the same compound at RT a_j in batch k_j and b_j in batch l_j | `ta`, `tb` |
| k_j, l_j | the two batches of anchor j | `ba`, `bb` |
| r_j | residual of anchor j (Eq. 3) | `r = X @ theta − y` |
| w_j | anchor weight (robust × bootstrap) | `w_rob`, `w0` |
| λ, λ′ = λ·N_eff/(KQ) | smoothness strength (raw, scaled) | `lam`, `lam_eff` |
| γ = 10³·N_eff/Q | gauge strength | `GAUGE_WEIGHT * n_eff / M` |
| N_eff | sum of anchor weights | `n_eff` |
| D₂ | second-difference matrix | `D2` (already multiplied by L) |
| G | gauge rows | `G_rows` |
| σ̂ | robust residual scale | `_robust_sigma(r)` |
| g_k→l = f_l⁻¹ ∘ f_k | map from batch k's clock to batch l's clock | `JointWarp.map(k, l, t)` |

## 0. Setup

We import NumPy (arrays), Matplotlib (plots) and the two constants the real code fixes: `N_BINS` (200) and `BIN_MIN` (0.225 minutes per bin). `np.set_printoptions` makes printed arrays short and readable.

`ipywidgets` gives the interactive sliders. If it is missing, each slider cell still draws a static plot.

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'tutorial' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))             # so we can import the real code in Section 18
from gcmsalign.config import BIN_MIN, N_BINS      # 0.225 min per bin, 200 bins

np.set_printoptions(precision=3, suppress=True, linewidth=110)
plt.rcParams.update({'figure.figsize': (8, 3.2), 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('N_BINS =', N_BINS, '  BIN_MIN =', BIN_MIN, 'min')

In [ ]:
try:
    import ipywidgets as W
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False


def with_slider(draw, name, options, default):
    """Draw once with the default value (a static figure, always shown). Then, if ipywidgets is
    installed, add a slider over `options` that redraws the figure."""
    draw(**{name: default})
    if HAVE_WIDGETS:
        W.interact(lambda v: draw(**{name: v}),
                   v=W.SelectionSlider(options=options, value=default, description=name))

print('ipywidgets available:', HAVE_WIDGETS)

## 1. The problem

**Intuition.** A gas chromatograph pushes a sample through a long column. Each compound comes out at its own *retention time* (RT), and a mass spectrometer records a *spectrum*: how much signal there is at each mass-to-charge (m/z) channel. So one run (one *chromatogram*) is a table: RT bins × m/z channels. Summing each row over m/z gives the *total ion chromatogram* (TIC), a single curve of signal against time.

Samples are measured in *batches*, on different days or months. Between batches the column ages and conditions change, so the **same compound comes out at a different time**. This is *drift*. If you compare two batches bin by bin, you compare different compounds, and every downstream statistic is wrong.

**Hand example.** Batch A shows a compound at bin 50. Batch B shows the same compound at bin 56. Comparing bin 50 with bin 50 compares the compound in A with background in B. We need a function that says "bin 56 in B is bin 50 in A".

**Why the toy uses T = 200.** The real fitter places its knots on `0 … N_BINS − 1` and inverts warps on a grid over the same range, with `N_BINS = 200` fixed in `gcmsalign.config`. To run the real code on the toy in Section 18, the toy uses the same 200 bins. Everything else is small: 4 batches, 2–5 samples each, 12 m/z channels and 10 compounds.

**The toy world.** Each compound has a fixed random spectrum shared by every batch. Batch k sees a compound whose *true* position is τ at h_k(τ) = τ + shift + stretch·(τ − 100) + a gentle Gaussian bump. Batch A is the reference, with h_A(τ) = τ. Batch D has only 2 samples, so it can later borrow strength from the others.

In [ ]:
T, M_MZ, N_COMP = N_BINS, 12, 10
NAMES, N_SAMPLES, REF = ['A', 'B', 'C', 'D'], [5, 4, 4, 2], 0
K = len(NAMES)
rng = np.random.default_rng(7)                       # fixed seed: same toy world every run

# A spectrum per compound: 12 m/z channels, about half of them zero, scaled to length 1.
spectra = rng.gamma(1.0, 1.0, (N_COMP, M_MZ)) * (rng.random((N_COMP, M_MZ)) < 0.5)
spectra /= np.linalg.norm(spectra, axis=1, keepdims=True)
tau = np.linspace(28, 172, N_COMP) + rng.uniform(-4, 4, N_COMP)   # true compound positions

# (shift, stretch, bump height, bump centre) per batch; A is the reference.
PARAMS = [(0, 0, 0, 100), (6, 0.04, 2.0, 80), (-5, -0.03, -2.0, 130), (9, 0.02, 3.0, 100)]

def h(k, x):
    """Where batch k sees a compound whose true (reference) position is x."""
    s, a, b, c = PARAMS[k]
    return x + s + a * (x - 100) + b * np.exp(-0.5 * ((x - c) / 25) ** 2)

print('true compound positions (bins):', tau.round(1))

`np.linspace(28, 172, 10)` makes 10 evenly spaced numbers from 28 to 172. `rng.uniform(-4, 4, 10)` adds a random nudge to each. `spectra /= np.linalg.norm(..., axis=1, keepdims=True)` divides every row by its own length. `axis=1` means "along each row", and `keepdims=True` keeps the result as a column, so the division lines up row by row. This is *broadcasting*: NumPy stretches the smaller array to match the bigger one.

Now we make the chromatograms. A compound is a Gaussian bump in time, 1.5 bins wide, multiplied by its spectrum. Each sample has small RT jitter (0.3 bins), a random height, and a 10 % chance of missing each compound.

In [ ]:
tt = np.arange(T)[:, None]          # column of bin indices, shape (200, 1)
positions, chromas = [], []         # positions[k][i, c] = where compound c sits in sample i of batch k
for k in range(K):
    P, C = [], []
    for _ in range(N_SAMPLES[k]):
        present = rng.random(N_COMP) > 0.1
        p = h(k, tau) + rng.normal(0, 0.3, N_COMP)                  # true position + jitter
        height = rng.lognormal(0, 0.3, N_COMP) * present
        profile = height * np.exp(-0.5 * ((tt - p) / 1.5) ** 2)     # (200, 10): one column per compound
        C.append(profile @ spectra + 1e-3 * rng.random((T, M_MZ)))  # (200, 12) chromatogram
        P.append(np.where(present, p, np.nan))
    positions.append(np.array(P))
    chromas.append(C)

tics = [[c.sum(axis=1) for c in cs] for cs in chromas]             # TIC = sum over m/z
print('one chromatogram has shape', chromas[0][0].shape, '(RT bins × m/z channels)')

`tt - p` subtracts a row of 10 positions from a column of 200 bins. Broadcasting turns that into a 200 × 10 table, one Gaussian per compound. `profile @ spectra` is a *matrix product* (Section 5 explains it): it mixes the 10 compound profiles with their spectra, giving a 200 × 12 chromatogram.

In [ ]:
grid = np.linspace(-50, 250, 3001)

def true_f(k, t):
    """True warp of batch k into the shared clock (= the reference clock): the inverse of h_k."""
    return np.interp(t, h(k, grid), grid)

def true_map(k, l, t):
    """True map from batch k's clock to batch l's clock."""
    return h(l, true_f(k, t))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for k in range(K):
    axes[0].plot(grid[350:2550], true_f(k, grid[350:2550]) - grid[350:2550], label=NAMES[k])
    axes[1].plot(np.arange(T), tics[k][0] + 1.2 * k, label=NAMES[k])
    axes[2].plot(np.arange(T), np.interp(h(k, np.arange(T)), np.arange(T), tics[k][0]) + 1.2 * k)
axes[0].set(xlabel='RT in the batch (bins)', ylabel='true drift d_k (bins)', title='True drifts')
axes[1].set(xlabel='RT (bins)', ylabel='TIC (offset per batch)', title='TICs as measured')
axes[2].set(xlabel='RT in the reference clock (bins)', title='TICs after the true warps')
axes[0].legend(); plt.tight_layout(); plt.show()

`np.interp(x, xp, fp)` does linear interpolation. Given points (xp, fp), with xp increasing, it returns the straight-line value at each x. Here `np.interp(t, h(k, grid), grid)` *inverts* h_k: it treats h_k's outputs as the x-coordinates. You will see this trick again for f⁻¹ in Section 12.

The drift is measured from the batch's clock *back to* the shared clock, d_k(t) = f_k(t) − t. Batch B sees compounds about 6 bins *late*, so its drift is negative (subtract about 6). Batch C sees them early, so its drift is positive.

In the middle panel, the peaks of different batches sit at different times. In the right panel they line up. That is what alignment must achieve, without knowing h_k.

In [ ]:
# Sanity checks on the toy world.
assert all(np.all(np.diff(h(k, grid)) > 0) for k in range(K)), 'every true warp must be monotone'
assert np.allclose(true_map(0, 0, tau), tau), 'reference maps to itself'
assert np.allclose(true_map(1, 0, h(1, tau)), tau, atol=1e-6), 'inverse works'
print('toy world OK')

**Check your understanding.** Where does batch D see the compound whose true position is 100 bins? Convert the answer to minutes.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
x = h(3, 100.0)
print(f"{x:.2f} bins = {x * BIN_MIN:.2f} min")   # shift 9 + bump 3 → 112 bins

## 2. A warp is just a function

**Intuition.** A *warp* f_k takes a time on batch k's clock and returns the time on a shared clock. If two batches see the same compound at a and b, a good pair of warps satisfies f_k(a) = f_l(b). The warp must be **monotone increasing**: if compound X elutes before compound Y in a batch, it must still come first on the shared clock. A warp that goes backwards would swap compounds.

We write the warp as identity plus drift, f_k(t) = t + d_k(t). Small drifts then mean small numbers, and "no drift" is d = 0.

**Hand example.** Take d(t) = 2 + 0.1t. Then f(10) = 10 + 3 = 13 and f(20) = 20 + 4 = 24. The slope is 1.1, positive, so f is monotone. With d(t) = −1.5t the slope is −0.5, so time would run backwards. Section 4 builds this rule into the parameters.

**Maths.** Monotone means f_k′(t) = 1 + d_k′(t) > 0. MBA asks for more, 0.5 ≤ f_k′(t) ≤ 2 (Eq. 2), so no batch runs more than twice as fast or slow as the shared clock.

In [ ]:
t = np.linspace(0, 199, 200)
for label, d in [('d(t) = 2 + 0.1 t', 2 + 0.1 * t), ('d(t) = −1.5 t', -1.5 * t)]:
    f = t + d
    slope = np.diff(f) / np.diff(t)          # np.diff: differences between neighbours
    print(f'{label:18s} slope ranges {slope.min():.2f} … {slope.max():.2f}, monotone: {np.all(slope > 0)}')
plt.plot(t, t + 2 + 0.1 * t, label='monotone warp'); plt.plot(t, t - 1.5 * t, label='not monotone')
plt.plot(t, t, 'k:', label='identity'); plt.xlabel('RT in the batch (bins)'); plt.ylabel('shared clock (bins)')
plt.legend(); plt.show()

`np.diff(f)` returns `f[1] − f[0], f[2] − f[1], …`, one shorter than `f`. Dividing by `np.diff(t)` gives the slope on each interval.

In [ ]:
assert np.all(np.diff(t + 2 + 0.1 * t) > 0) and not np.all(np.diff(t - 1.5 * t) > 0)
print("monotonicity check OK")

**Check your understanding.** Is f(t) = t + 5 sin(t / 3) monotone on [0, 199]? Check its slope numerically.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
s = np.diff(t + 5 * np.sin(t / 3)) / np.diff(t)
print("min slope", s.min().round(2), "→ monotone?", bool(np.all(s > 0)))   # slope 1 + (5/3)cos(t/3) dips below 0

## 3. Piecewise-linear drift and the hat basis

**Intuition.** We can't learn an arbitrary function. MBA describes each drift by its values at Q equally spaced *knots*, and joins them with straight lines. Between two knots the drift is a weighted average of the two knot values. The closer t is to a knot, the more that knot counts.

**Vectors and matrices.** A *vector* is a list of numbers, like (0, 2, 1, 3). A *matrix* is a table of numbers. A *matrix–vector product* Bv takes each row of B, multiplies it element by element with v, and adds up. For example, the row (0, ½, ½, 0) times (0, 2, 1, 3) is 0·0 + ½·2 + ½·1 + 0·3 = 1.5.

**Hand example.** Knots u = (0, 10, 20, 30), drift values at the knots d = (0, 2, 1, 3). At t = 15 we are halfway between knots 1 and 2, so the weights are B(15) = (0, ½, ½, 0) and d(15) = 1.5. At t = 22 we are 0.2 of the way from knot 2 to knot 3, so B(22) = (0, 0, 0.8, 0.2) and d(22) = 0.8·1 + 0.2·3 = 1.4.

**Maths.** d_k(t) = B(t) · (d_k at the knots). B(t) is a row with at most two non-zero entries, which sum to 1. Stacking the rows for many times gives a matrix B, and **interpolation becomes a matrix–vector product**. That is what makes the whole fit linear.

| Symbol | Code |
|---|---|
| knots u | `knots` |
| B(t) | `_hat_basis(t, knots)` |
| left knot index | `idx` |
| fraction inside the interval | `frac` |

In [ ]:
def hat_basis(t, knots):
    """B: one row per time in t, one column per knot; linear-interpolation weights."""
    t = np.asarray(t, dtype=float)
    idx = np.clip(np.searchsorted(knots, t, side='right') - 1, 0, len(knots) - 2)   # left knot
    frac = np.clip((t - knots[idx]) / (knots[idx + 1] - knots[idx]), 0.0, 1.0)      # 0 … 1
    B = np.zeros((len(t), len(knots)))
    B[np.arange(len(t)), idx] = 1.0 - frac          # weight on the left knot
    B[np.arange(len(t)), idx + 1] = frac            # weight on the right knot
    return B

u = np.array([0., 10, 20, 30]); d_knots = np.array([0., 2, 1, 3])
B = hat_basis([15, 22], u)
print(B); print('B @ d =', B @ d_knots)

- `np.searchsorted(knots, t, side='right') − 1` finds, for each t, the index of the knot just to its left.
- `np.clip(x, lo, hi)` caps values into [lo, hi]. The real code clips too, so **times outside the knot range are clamped to the end knots**, a detail that matters near the run's ends.
- `B[np.arange(n), idx] = …` is *fancy indexing*: row i gets its value in column `idx[i]`.

In [ ]:
ts = np.linspace(0, 30, 301)
plt.figure(figsize=(8, 3))
for q in range(len(u)):
    plt.plot(ts, hat_basis(ts, u)[:, q], label=f'weight of knot {q}')
plt.plot(ts, hat_basis(ts, u) @ d_knots, 'k', lw=2, label='drift = B @ d')
plt.xlabel('RT (bins)'); plt.ylabel('weight / drift (bins)'); plt.legend(ncol=5, fontsize=8); plt.show()

Each coloured "hat" is one column of B. The black line is the drift: a weighted sum of the hats.

In [ ]:
assert np.allclose(hat_basis(ts, u) @ d_knots, np.interp(ts, u, d_knots)), 'B @ d must equal np.interp'
assert np.allclose(hat_basis(ts, u).sum(axis=1), 1.0), 'weights sum to 1'
assert np.allclose(B @ d_knots, [1.5, 1.4])
print('hat basis matches np.interp')

**Check your understanding.** Build B(t) for t = 5 by hand on the knots u, then check it with `hat_basis`. What is d(5)?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
print(hat_basis([5], u), hat_basis([5], u) @ d_knots)   # (0.5, 0.5, 0, 0) → d(5) = 1.0

## 4. Parameterising by increments

**Intuition.** Instead of storing the drift at each knot, MBA stores the drift at the *first* knot, c, plus the *increments* δ between neighbouring knots. You get the knot values back by adding up: d_0 = c, d_1 = c + δ_1, d_2 = c + δ_1 + δ_2, …

Why bother? Because the slope of f on the interval between two knots is 1 + δ/Δu. **A bound on the slope becomes a bound on a single number**, δ. That is a *box bound*, which the solver handles easily.

**Hand example.** θ = (c, δ_1, δ_2, δ_3) = (0, 2, −1, 2). Running sums give d = (0, 2, 1, 3), the drift from Section 3. With Δu = 10, the slope on interval 1 is 1 + 2/10 = 1.2. The rule 0.5 ≤ slope ≤ 2 becomes −5 ≤ δ ≤ 10.

**Maths.** d_k(u) = Lθ_k, where L is the *lower-triangular* matrix of ones (ones on and below the diagonal). Multiplying by L takes running sums, like `np.cumsum`. The bounds (Eq. 2) are

  (0.5 − 1)·Δu ≤ δ_kq ≤ (2 − 1)·Δu,  c_k unbounded.

| Symbol | Code |
|---|---|
| L | `Lcum = np.tril(np.ones((M, M)))` |
| bounds | `lb`, `ub`, from `S_MIN, S_MAX = 0.5, 2.0` |

In [ ]:
Q4 = 4
L4 = np.tril(np.ones((Q4, Q4)))            # np.tril keeps the lower triangle, zeros above
theta = np.array([0., 2, -1, 2])
print(L4); print('L @ theta =', L4 @ theta, '  np.cumsum(theta) =', np.cumsum(theta))
du = np.diff(u)
print('slopes 1 + delta/du =', 1 + theta[1:] / du)
lb4 = np.r_[-np.inf, (0.5 - 1) * du]; ub4 = np.r_[np.inf, (2 - 1) * du]   # np.r_ joins pieces
print('lower bounds', lb4, '\nupper bounds', ub4)

In [ ]:
assert np.allclose(L4 @ theta, np.cumsum(theta)) and np.allclose(L4 @ theta, d_knots)
assert np.allclose(1 + theta[1:] / du, np.diff(L4 @ theta + u) / du), 'slope of f = 1 + δ/Δu'
print('increments check OK')

**Check your understanding.** Which increments δ would make the warp exactly the identity on every interval? And which would make it twice as fast?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
print("identity: delta = 0;  twice as fast: delta = du =", du)   # slope 1 + du/du = 2, the upper bound

## 5. Least squares from zero

**Intuition.** We have more equations than unknowns, and they can't all hold exactly because the data is noisy. *Least squares* picks the unknowns that make the sum of squared errors as small as possible.

**Hand example: a line through 3 points.** Points (0, 1), (1, 3), (2, 4), model y = m·x + c. As a matrix equation Xβ ≈ y:

X = [[0, 1], [1, 1], [2, 1]] (one row per point: x and a constant 1), β = (m, c), y = (1, 3, 4).

The *transpose* Xᵀ flips rows and columns. The best β solves the *normal equations* XᵀXβ = Xᵀy:

XᵀX = [[5, 3], [3, 3]], Xᵀy = (11, 8).

So 5m + 3c = 11 and 3m + 3c = 8. Subtracting gives 2m = 3, so m = 1.5 and c = 1.1667.

**Why a matrix can be singular.** If two columns of X carry the same information, many β fit equally well, so there is no unique answer. For X = [[1, 1], [2, 2]] the columns are equal, XᵀX = [[5, 5], [5, 5]] has determinant 0, and it can't be inverted. Section 8 meets exactly this problem.

In [ ]:
X5 = np.array([[0., 1], [1, 1], [2, 1]]); y5 = np.array([1., 3, 4])
XtX, Xty = X5.T @ X5, X5.T @ y5              # .T is the transpose
print('XᵀX =\n', XtX, '\nXᵀy =', Xty)
beta_normal = np.linalg.solve(XtX, Xty)       # solve the normal equations
beta_lstsq = np.linalg.lstsq(X5, y5, rcond=None)[0]   # NumPy's least-squares solver
print('normal equations:', beta_normal, '  lstsq:', beta_lstsq)
print('determinant of the singular example:', np.linalg.det(np.array([[5., 5], [5, 5]])))
xs = np.linspace(-0.2, 2.2, 10)
plt.plot(X5[:, 0], y5, 'o', label='data'); plt.plot(xs, beta_normal[0] * xs + beta_normal[1], label='least-squares line')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.show()

In [ ]:
assert np.allclose(beta_normal, [1.5, 7 / 6]) and np.allclose(beta_normal, beta_lstsq)
print("least squares OK")

**Check your understanding.** Add a fourth point (3, 7) and refit. Does the slope go up or down?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
X = np.array([[0., 1], [1, 1], [2, 1], [3, 1]]); y = np.array([1., 3, 4, 7])
print(np.linalg.lstsq(X, y, rcond=None)[0])   # slope rises to 1.9

## 6. One batch against a fixed reference

**Intuition.** Start with the simplest case: batch B against the reference A, whose drift is fixed at zero. Each anchor says "this compound is at a in A and at b in B". With f_A(a) = a, the equation f_A(a) = f_B(b) becomes a = b + d_B(b), so d_B(b) = a − b. One linear equation per anchor, unknowns θ_B.

**Perfect anchors.** Before harvesting anchors from spectra (Section 14), we use the true peak positions: for every pair of samples from two different batches, every compound present in both gives one anchor.

**Maths.** d_B(b) = B(b)Lθ_B, so each anchor gives a row B(b_j)L and a target a_j − b_j. Plain least squares on those rows recovers θ_B.

In [ ]:
rows = []                       # one row per anchor: (batch k, batch l, sample i, sample j, a, b)
for ka, kb in combinations(range(K), 2):
    for i in range(N_SAMPLES[ka]):
        for j in range(N_SAMPLES[kb]):
            for c in range(N_COMP):
                pa, pb = positions[ka][i, c], positions[kb][j, c]
                if np.isfinite(pa) and np.isfinite(pb):
                    rows.append((ka, kb, i, j, pa, pb))
anc = np.array(rows)
ba, bb, sa, sb = (anc[:, q].astype(int) for q in range(4))
ta, tb = anc[:, 4], anc[:, 5]
print(len(ta), 'perfect anchors; per batch pair:',
      {f'{NAMES[x]}–{NAMES[y]}': int(np.sum((ba == x) & (bb == y))) for x, y in combinations(range(K), 2)})

`combinations(range(4), 2)` lists each pair of batches once: (0,1), (0,2), …, (2,3). The anchor table uses the same columns as the real `Anchors` class: `ba, bb, sa, sb, ta, tb`.

In [ ]:
Q = 11                                         # knots for the toy (the real default is 41)
knots = np.linspace(0, T - 1, Q); L = np.tril(np.ones((Q, Q)))
m_AB = (ba == 0) & (bb == 1)                   # anchors between A (ref) and B
X_B = hat_basis(tb[m_AB], knots) @ L           # rows B(b) L
y_B = ta[m_AB] - tb[m_AB]                      # targets a − b
theta_B = np.linalg.lstsq(X_B, y_B, rcond=None)[0]
d_B = L @ theta_B
tg = np.linspace(35, 175, 141)
plt.plot(tg, true_f(1, tg) - tg, 'k--', label='true drift of B')
plt.plot(tg, np.interp(tg, knots, d_B), label='fitted from A–B anchors')
plt.plot(tb[m_AB], y_B, '.', alpha=0.3, label='anchors (a − b)')
plt.xlabel('RT in batch B (bins)'); plt.ylabel('drift (bins)'); plt.legend(); plt.show()

In [ ]:
err6 = np.abs(np.interp(tg, knots, d_B) - (true_f(1, tg) - tg)).max()
print(f'max error vs the true drift: {err6:.3f} bins')
assert err6 < 0.6, 'one-batch fit should recover the drift to within 0.6 bins'

**Check your understanding.** How many anchors does B share with A? Refit using only the first 10 of them. Is the error larger?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
Xs, ys = X_B[:10], y_B[:10]
d10 = L @ np.linalg.lstsq(Xs, ys, rcond=None)[0]
print(np.abs(np.interp(tg, knots, d10) - (true_f(1, tg) - tg)).max())   # much larger: knots with no anchors are guesses

## 7. Many batches jointly: the design matrix

**Intuition.** Now every batch has unknown drift, and an anchor between batches k and l involves *both*. Put all parameters in one long vector θ = (θ_0, θ_1, …, θ_{K−1}). Each block has Q numbers, so θ has KQ entries. Every anchor becomes one row of a big *design matrix* X.

**Block matrices.** A block matrix is a matrix cut into tiles. Here the columns are cut into K tiles of width Q, one per batch.

**Maths (Eq. 3).** The residual of anchor j is

  r_j = a_j − b_j + B(a_j)Lθ_{k_j} − B(b_j)Lθ_{l_j}.

As a row: put **+B(a_j)L in batch k_j's tile** and **−B(b_j)L in batch l_j's tile**, zeros elsewhere, with target y_j = b_j − a_j. Then r = Xθ − y. It is linear in θ, so least squares applies.

**Hand example.** K = 2, Q = 3, knots (0, 100, 200). An anchor with a = 50 in batch 0 and b = 150 in batch 1: B(50)L = (1, 0.5, 0) and B(150)L = (1, 1, 0.5). The row is (1, 0.5, 0, −1, −1, −0.5) and y = 100.

In [ ]:
kn3, L3 = np.array([0., 100, 200]), np.tril(np.ones((3, 3)))
row = np.r_[hat_basis([50], kn3) @ L3, -(hat_basis([150], kn3) @ L3)].ravel()
print('hand example row:', row)

def design_matrix(ba, bb, ta, tb, K, knots):
    """X (one row per anchor, K*Q columns) and y = tb − ta, as in fit_joint."""
    Q = len(knots); L = np.tril(np.ones((Q, Q)))
    BaL, BbL = hat_basis(ta, knots) @ L, hat_basis(tb, knots) @ L
    X = np.zeros((len(ta), K * Q))
    for j in range(len(ta)):
        X[j, ba[j] * Q:(ba[j] + 1) * Q] += BaL[j]     # +B(a)L in batch k's columns
        X[j, bb[j] * Q:(bb[j] + 1) * Q] -= BbL[j]     # −B(b)L in batch l's columns
    return X, tb - ta

X, y = design_matrix(ba, bb, ta, tb, K, knots)
print('X shape', X.shape, '(anchors × K·Q)')

`X[j, k*Q:(k+1)*Q]` selects row j and the Q columns of batch k: a *slice*. `.ravel()` flattens a table into one row.

In [ ]:
order = np.lexsort((bb, ba))                       # sort rows by batch pair, for a tidy picture
plt.figure(figsize=(8, 4))
plt.imshow(np.sign(X[order]), aspect='auto', cmap='coolwarm', interpolation='nearest')
for k in range(1, K):
    plt.axvline(k * Q - 0.5, color='k', lw=0.8)
plt.xticks([k * Q + Q / 2 for k in range(K)], [f'batch {n} θ' for n in NAMES])
plt.ylabel('anchor (sorted by batch pair)'); plt.title('Design matrix: red +B(a)L, blue −B(b)L, grey 0')
plt.show()

In [ ]:
th_true = np.zeros(K * Q)                  # true θ at the knots (reference gauge: A has zero drift)
for k in range(K):
    th_true[k * Q:(k + 1) * Q] = np.r_[true_f(k, knots[:1]) - knots[:1], np.diff(true_f(k, knots) - knots)]
print('residuals at the true θ: median |r| =', np.median(np.abs(X @ th_true - y)).round(3), 'bins')
assert np.allclose(row, [1, 0.5, 0, -1, -1, -0.5]) and np.median(np.abs(X @ th_true - y)) < 0.6

At the true drifts the residuals are small but not zero. The jitter we added (0.3 bins) and the piecewise-linear approximation both leave a little error.

**Check your understanding.** Which tiles of X are non-zero for an anchor between batches B and D? Check with the first such anchor.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
j = np.flatnonzero((ba == 1) & (bb == 3))[0]
print([bool(np.any(X[j, k*Q:(k+1)*Q])) for k in range(K)])   # only tiles B and D

## 8. The gauge problem

**Intuition.** The anchors only say where batches sit *relative to each other*. Move the whole shared clock by one bin (add 1 to every batch's drift) and every anchor equation still holds. Nothing in the data can pin that down, so the solution isn't unique. This is the *gauge* freedom.

**Hand example.** Anchor a = 50 (batch 0), b = 56 (batch 1), drifts d_0 = 0 and d_1 = −6: r = 50 − 56 + 0 − (−6) = 0. Add 1 to both drifts: r = 50 − 56 + 1 − (−5) = 0. Same residual.

**A note against the summary.** It is only a common **constant** that leaves every residual exactly unchanged. Adding a common *non-constant* function g to every drift changes r_j by g(a_j) − g(b_j), which is not zero when a_j ≠ b_j. The code's gauge term handles the constant (and steadies the near-flat directions around it).

**Maths.** The constant shift is the vector v with c_k = 1 for every k and all δ = 0. Xv = 0, so XᵀX is singular. Two fixes, both written as penalty rows γ‖Gθ‖²:
- **reference gauge:** G = L in the reference batch's tile, pushing that batch's drift to 0 at every knot;
- **mean gauge:** G = L in every tile, pushing Σ_k d_k = 0 at every knot.

γ = 10³·N_eff/Q is large, so these are *nearly* exact constraints. The code adds them as penalties, not hard constraints, so the reference drift is pinned only approximately.

In [ ]:
v = np.zeros(K * Q); v[::Q] = 1.0                  # c_k += 1 for every batch (v[::Q] = every Q-th entry)
print('|X v| =', np.abs(X @ v).max(), '  (a common constant changes no residual)')
g = np.zeros(K * Q)                                # add the ramp t/100 to every drift:
for k in range(K):                                 # every increment δ grows by Δu/100
    g[k * Q + 1:(k + 1) * Q] = np.diff(knots) / 100
print('|X g| =', np.abs(X @ g).max().round(3), '  (a common ramp changes residuals by (a − b)/100)')
s = np.linalg.svd(X, compute_uv=False)             # singular values (Section 5's 'determinant 0', in general)
print('rank of X:', np.linalg.matrix_rank(X), 'of', K * Q, '   smallest singular values:', [f'{x:.1e}' for x in s[-3:]])

The rank is below K·Q, so X has *several* null directions, not just one. The constant shift is one. The others come from knots no anchor touches: our compounds sit between about 26 and 180 bins, so the end knots at 0 and 199 are unconstrained. The gauge fixes the first, and the smoothness penalty (Section 9) fixes the others. A *singular value* measures how much X stretches a direction, and a zero means X ignores that direction entirely.

In [ ]:
def gauge_rows(gauge, K, Q):
    """G: Q rows, K*Q columns. gauge = 'mean' or a batch index (reference gauge)."""
    L = np.tril(np.ones((Q, Q))); G = np.zeros((Q, K * Q))
    for k in (range(K) if gauge == 'mean' else [gauge]):
        G[:, k * Q:(k + 1) * Q] = L
    return G

def penalised_lstsq(X, y, extra_rows):
    """Least squares with penalty rows stacked under X (targets 0)."""
    A = np.vstack([X] + extra_rows)
    return np.linalg.lstsq(A, np.r_[y, np.zeros(A.shape[0] - len(y))], rcond=None)[0]

n_eff, gam = len(y), 1e3 * len(y) / Q
fits = {g_: penalised_lstsq(X, y, [np.sqrt(gam) * gauge_rows(g_, K, Q), 1e-3 * np.eye(K * Q)])
        for g_ in (REF, 'mean')}

`np.vstack` stacks matrices on top of each other. A penalty term γ‖Gθ‖² is the same as extra rows √γ·G with target 0, because their squared residuals add γ‖Gθ‖² to the sum. That is the *augmented least squares* trick. The tiny `1e-3 * I` rows stand in for smoothness until Section 9, so the unanchored end knots get a definite value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, g_ in zip(axes, (REF, 'mean')):
    th = fits[g_]
    for k in range(K):
        ax.plot(knots, L @ th[k * Q:(k + 1) * Q], 'o-', label=NAMES[k])
    ax.set(xlabel='RT (bins)', title=f'gauge = {g_!r}' + (' (reference A)' if g_ == REF else ' (Σ d_k = 0)'))
axes[0].set_ylabel('fitted drift at the knots (bins)'); axes[0].legend(); plt.show()
dr = {g_: np.stack([L @ fits[g_][k * Q:(k + 1) * Q] for k in range(K)]) for g_ in fits}
print(f'reference gauge: |d_A| max = {np.abs(dr[REF][0]).max():.1e} bins  (a penalty, so not exactly 0)')
print(f'mean gauge:      |Σ d_k| max = {np.abs(dr["mean"].sum(0)).max():.1e} bins')

The two gauges put the shared clock in different places, so the drifts themselves differ, and so does "B's drift minus A's drift" at the same time. What we actually use is the **map between two batches**, f_A⁻¹(f_k(t)), which takes a time in batch k to the time in A. We invert f_A with `np.interp`, swapping its x and y, as in Section 1. Section 12 builds this properly.

Do the gauges give the same maps? **Nearly, but not exactly.** A pure shift of the clock changes nothing. But each drift is piecewise linear, and smoothed, *in its own batch's time*. Moving the shared clock by anything other than a shift changes which warps count as smooth, so the two gauges settle on slightly different solutions. Here the maps differ by about 0.2 bins (the real fitter gives the same, and the gap grows with λ). That is why the paper lists the choice of gauge as an open question.

In [ ]:
def map_to_A(D, k, t):
    fine = np.linspace(0, T - 1, 4 * T)
    fA = fine + np.interp(fine, knots, D[0])                     # f_A on a fine grid
    return np.interp(t + np.interp(t, knots, D[k]), fA, fine)    # f_A⁻¹(f_k(t))

tg8 = np.linspace(40, 160, 61)
diff8 = max(np.abs(map_to_A(dr[REF], k, tg8) - map_to_A(dr['mean'], k, tg8)).max() for k in range(1, K))
print('maps to A differ by at most', round(diff8, 4), 'bins between the two gauges')
assert np.abs(dr[REF][0]).max() < 0.01 and np.abs(dr['mean'].sum(0)).max() < 0.01
assert diff8 < 0.3, 'the gauges should give maps within a fraction of a bin of each other'

**Check your understanding.** Fit with no gauge rows at all (only the tiny `1e-3 * I` rows). Is A's drift still near 0? Why not?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
th0 = penalised_lstsq(X, y, [1e-3 * np.eye(K * Q)])
print("A drift:", (L @ th0[:Q]).round(2))   # whatever the tiny ridge prefers: nothing in the data fixes the clock

## 9. The smoothness penalty

**Intuition.** With many knots, a drift can wiggle to chase noise. *Regularisation* adds a cost for complexity. Here the cost is for *bending*: a straight line is free, and every kink costs. λ sets how much we care. Too small and the drift chases noise; too large and it can only be a straight line.

**Hand example.** The second difference of (0, 2, 1, 3) is (0 − 2·2 + 1, 2 − 2·1 + 3) = (−3, 3). Each entry measures how much the line bends at a knot. For a straight line, like (0, 1, 2, 3), every second difference is 0.

**Maths (Eq. 4).** Penalty λ′ Σ_k ‖D₂Lθ_k‖², with D₂ the second-difference matrix (rows of (1, −2, 1)). The code scales λ as λ′ = λ·N_eff/(KQ): the penalty grows with the number of anchors, so a given λ means the same smoothness whether there are 600 anchors or 25,000. As augmented rows: √λ′·D₂L in each batch's tile.

| Symbol | Code |
|---|---|
| D₂L | `D2 = np.diff(np.eye(M), n=2, axis=0) @ Lcum` |
| λ′ | `lam_eff = lam * n_eff / (K * M)` |

In [ ]:
print('second differences of (0,2,1,3):', np.diff(d_knots, n=2))
D2 = np.diff(np.eye(Q), n=2, axis=0) @ L          # (Q−2) × Q; acts on θ_k directly

def smooth_rows(lam, n_eff, K, Q):
    """√λ′ · D₂L in every batch's tile (block-diagonal)."""
    D2 = np.diff(np.eye(Q), n=2, axis=0) @ np.tril(np.ones((Q, Q)))
    S = np.zeros(((Q - 2) * K, K * Q))
    for k in range(K):
        S[k * (Q - 2):(k + 1) * (Q - 2), k * Q:(k + 1) * Q] = D2
    return np.sqrt(lam * n_eff / (K * Q)) * S
print('D₂L applied to the true θ_B gives the bends of B\'s true drift:', (D2 @ th_true[Q:2 * Q]).round(2))

`np.diff(np.eye(Q), n=2, axis=0)` takes second differences of the rows of the identity matrix, which builds the (1, −2, 1) pattern. Multiplying by L lets it act on increments θ rather than knot values.

In [ ]:
Q9 = 21                                            # more knots, so wiggles are possible
kn9 = np.linspace(0, T - 1, Q9); L9 = np.tril(np.ones((Q9, Q9)))
X9, y9 = design_matrix(ba, bb, ta, tb, K, kn9)
noisy = y9 + np.random.default_rng(1).normal(0, 1.5, len(y9))   # extra anchor noise, to tempt wiggles

def fit_smooth(lam, Xm=X9, ym=noisy, Qm=Q9):
    G = np.sqrt(1e3 * len(ym) / Qm) * gauge_rows(REF, K, Qm)
    return penalised_lstsq(Xm, ym, [G, smooth_rows(lam, len(ym), K, Qm), 1e-4 * np.eye(K * Qm)])

def draw_lambda(lam):
    th = fit_smooth(lam); tg = np.linspace(30, 175, 300)
    plt.figure(figsize=(8, 3))
    for k in (1, 3):
        plt.plot(tg, true_f(k, tg) - tg, 'k--', lw=1)
        plt.plot(tg, np.interp(tg, kn9, L9 @ th[k * Q9:(k + 1) * Q9]), label=f'batch {NAMES[k]}')
    plt.xlabel('RT (bins)'); plt.ylabel('drift (bins)'); plt.title(f'λ = {lam}   (dashed: truth)')
    plt.legend(); plt.show()

with_slider(draw_lambda, 'lam', [1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100, 1e3, 1e4], 1.0)

In [ ]:
def max_err(th, Qm, kn, ks=(1, 2, 3)):
    tg = np.linspace(35, 170, 200); Lm = np.tril(np.ones((Qm, Qm)))
    return max(np.abs(np.interp(tg, kn, Lm @ th[k * Qm:(k + 1) * Qm]) - (true_f(k, tg) - tg)).max() for k in ks)

errs9 = {lam: max_err(fit_smooth(lam), Q9, kn9) for lam in (1e-4, 1.0, 1e4)}
print({k: round(v, 2) for k, v in errs9.items()}, '(max error vs truth, bins)')
assert errs9[1.0] < errs9[1e-4] and errs9[1.0] < errs9[1e4], 'a middle λ should beat both extremes'

Too little smoothing chases the extra noise. Too much forces a straight line that misses the bumps. Section 15 picks λ from the data, without looking at the truth.

**Check your understanding.** Double every anchor (stack X9 and noisy on themselves) and refit at λ = 1. Does the fit change? Why does that show the point of the N_eff/(KQ) scaling?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
th1 = fit_smooth(1.0); th2 = fit_smooth(1.0, np.vstack([X9, X9]), np.r_[noisy, noisy])
print(np.abs(th1 - th2).max())   # ~0: λ′ grows with N_eff, so duplicating the data leaves the balance unchanged

## 10. Bounds: keeping every warp monotone

**Intuition.** With few, noisy anchors and little smoothing, nothing stops the fit from making a batch's clock run backwards for a stretch, if that fits two mismatched anchors better. Section 4 turned "slope between 0.5 and 2" into simple bounds on each increment δ. A *bounded* least-squares solver finds the best fit that respects them.

**Hand example.** With knot spacing Δu = 9.95 bins, the bounds are −4.98 ≤ δ ≤ 9.95. An unbounded fit that wants δ = −30 on one interval (slope 1 − 30/9.95 = −2.0, time running backwards) gets δ = −4.98 instead (slope 0.5), and the other parameters adjust around it.

**Maths.** minimise ‖Aθ − b‖² subject to lb ≤ θ ≤ ub. Here A is X with the penalty rows stacked under it. `scipy.optimize.lsq_linear(A, b, bounds=(lb, ub), method='bvls')` solves it. BVLS (bounded-variable least squares) is an *active-set* method: it fixes the parameters that would break a bound at that bound, and solves ordinary least squares for the rest, repeating until nothing changes.

**How the real code solves it.** `fit_joint` doesn't stack rows. It forms the normal matrix G = XᵀWX + P, takes its Cholesky factor C (G = CCᵀ), and calls `lsq_linear(C.T, C⁻¹XᵀWy, bounds, method='bvls')`. Both routes minimise exactly the same quadratic, so they give the same answer. Section 18 checks this to about 10⁻¹³ bins.

| Symbol | Code |
|---|---|
| lb, ub | `lb = np.tile(np.concatenate([[-np.inf], -(1 - S_MIN) * dk]), K)` and `ub` likewise |
| bounded solve | `lsq_linear(C.T, z, bounds=(lb, ub), method='bvls')` |

In [ ]:
from scipy.optimize import lsq_linear

def build_problem(ba, bb, ta, tb, K, lam, gauge, Q, w0):
    """Everything fit_joint builds before its solve: X, y, the stacked penalty rows, bounds."""
    knots = np.linspace(0, T - 1, Q); du = np.diff(knots)
    X, y = design_matrix(ba, bb, ta, tb, K, knots)
    n_eff = max(float(w0.sum()), 1.0)
    pen = np.vstack([np.sqrt(1e3 * n_eff / Q) * gauge_rows(gauge, K, Q),   # γ = 10³·N_eff/Q
                     smooth_rows(lam, n_eff, K, Q),                       # λ′ = λ·N_eff/(KQ)
                     np.sqrt(1e-8 * n_eff) * np.eye(K * Q)])              # the code's tiny ridge
    lb = np.tile(np.r_[-np.inf, (0.5 - 1) * du], K)                       # c unbounded, δ ≥ −0.5·Δu
    ub = np.tile(np.r_[np.inf, (2.0 - 1) * du], K)                        # δ ≤ +1.0·Δu
    return knots, X, y, pen, lb, ub

def solve(X, y, pen, w, lb, ub, bounded=True):
    """Weighted, penalised, (optionally) bounded least squares by stacking rows."""
    A = np.vstack([np.sqrt(w)[:, None] * X, pen])
    b = np.r_[np.sqrt(w) * y, np.zeros(len(pen))]
    bounds = (lb, ub) if bounded else (-np.inf, np.inf)
    return lsq_linear(A, b, bounds=bounds, method='bvls').x

`np.tile(v, K)` repeats a vector K times, giving one copy of the bounds per batch. `np.sqrt(w)[:, None] * X` multiplies each row of X by √w_j. Broadcasting again: `[:, None]` turns the weights into a column. Weighting rows by √w gives Σ w_j r_j² after squaring.

The ridge `1e-8 · N_eff · I` is **not in the paper**. The code adds it so the normal matrix can always be factorised. It is far too small to change a fit you could see.

In [ ]:
# Two crossing anchors between A and D (compound order swapped), plus a few clean ones.
tA = np.array([40., 58, 64, 90, 120, 150]); tD = h(3, tA)
tD[1], tD[2] = tD[1] + 5, tD[2] - 5                    # the 58 and 64 anchors now cross
z = np.zeros(6, int)
Q10 = 21
slopes = {}
for bounded in (False, True):
    kn, X10, y10, pen10, lb10, ub10 = build_problem(z, z + 1, tA, tD, 2, 1e-3, 0, Q10, np.ones(6))
    th = solve(X10, y10, pen10, np.ones(6), lb10, ub10, bounded)
    f_D = kn + np.tril(np.ones((Q10, Q10))) @ th[Q10:]
    slopes[bounded] = np.diff(f_D) / np.diff(kn)
    plt.plot(kn, f_D - kn, 'o-', label=('bounded' if bounded else 'unbounded') + f' (min slope {slopes[bounded].min():.2f})')
plt.plot(tD, tA - tD, 'kx', ms=8, label='anchors (a − b)')
plt.xlabel('RT in batch D (bins)'); plt.ylabel('drift of D (bins)'); plt.legend(); plt.show()

In [ ]:
assert slopes[False].min() < 0, 'the unbounded fit should run backwards somewhere'
assert slopes[True].min() >= 0.5 - 1e-9 and slopes[True].max() <= 2 + 1e-9, 'bounded slopes stay in [0.5, 2]'
print('unbounded min slope', slopes[False].min().round(2), '→ bounded', slopes[True].min().round(2))

**Check your understanding.** With the bounded fit, on how many knot intervals is a bound *active*, that is, the slope exactly 0.5 or 2?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
print(np.sum(np.isclose(slopes[True], 0.5)) + np.sum(np.isclose(slopes[True], 2.0)), "active bounds")

## 11. Robust reweighting

**Intuition.** Real anchors include mistakes: two different compounds with similar spectra get matched. One wrong anchor 30 bins off adds 900 to the sum of squares, so it can drag the whole fit. A *robust loss* listens less to points that are far off. MBA does this by *iteratively reweighted least squares* (IRLS): fit, look at the residuals, give large residuals smaller weights, fit again.

**The scale σ̂.** To call a residual "large" we need a typical size that the outliers themselves can't inflate. The *median absolute deviation* (MAD) is the median of |r − median(r)|. For normal data the standard deviation is about 1.4826 × MAD. The code floors it at 0.25 bins: σ̂ = max(1.4826·MAD, 0.25).

**Hand example.** Residuals (1, 2, 3, 4, 100). The median is 3, the absolute deviations are (2, 1, 0, 1, 97), so MAD = 1 and σ̂ = 1.4826. The standard deviation would be 39, ruined by the single outlier.

**Two weight functions.**
- **Huber**, 3 rounds, c = 1.345σ̂: w = 1 if |r| ≤ c, else c/|r|. Big residuals count less but never zero.
- **Tukey bisquare**, 4 rounds, c = 4.685σ̂: w = (1 − (r/c)²)² if |r| < c, else 0. Gross outliers are dropped entirely.

Huber first gives a stable starting fit, and Tukey then removes what is clearly wrong. With σ̂ = 1.4826, the residual 100 gets Huber weight 1.994/100 ≈ 0.02 and Tukey weight 0.

**Order of operations in the code.** 7 rounds of (solve → residuals → σ̂ → new weights), then one final solve with the last weights: 8 solves in all. σ̂ uses only anchors with positive bootstrap weight (`r[w0 > 0]`).

In [ ]:
def robust_sigma(r):
    return max(1.4826 * float(np.median(np.abs(r - np.median(r)))), 0.25) if len(r) else np.nan

r_hand = np.array([1., 2, 3, 4, 100])
sig = robust_sigma(r_hand); c_h, c_t = 1.345 * sig, 4.685 * sig
print('σ̂ =', round(sig, 4), '  std =', r_hand.std().round(1))
print('Huber weights', np.where(np.abs(r_hand) <= c_h, 1, c_h / np.abs(r_hand)).round(3))
print('Tukey weights', np.where(np.abs(r_hand) < c_t, (1 - (r_hand / c_t) ** 2) ** 2, 0).round(3))
assert np.isclose(sig, 1.4826)

Note that Tukey also down-weights the *good* residuals of 3 and 4 a little, because they are not tiny relative to c. That is expected: only residuals near 0 keep weight ≈ 1.

In [ ]:
def toy_fit(ba, bb, ta, tb, K, lam, gauge, Q, w0=None, n_huber=3, n_tukey=4, bounded=True):
    """The whole MBA fit, from scratch. Mirrors multibatch.fit_joint step by step."""
    w0 = np.ones(len(ta)) if w0 is None else np.asarray(w0, dtype=float)
    knots, X, y, pen, lb, ub = build_problem(ba, bb, ta, tb, K, lam, gauge, Q, w0)
    w_rob, history = np.ones(len(ta)), []
    for it in range(n_huber + n_tukey):
        theta = solve(X, y, pen, w0 * w_rob, lb, ub, bounded)
        r = X @ theta - y
        sig = robust_sigma(r[w0 > 0]); a = np.abs(r)
        if it < n_huber:
            c = 1.345 * sig; w_rob = np.where(a <= c, 1.0, c / np.maximum(a, 1e-12))
        else:
            c = 4.685 * sig; w_rob = np.where(a < c, (1 - (a / c) ** 2) ** 2, 0.0)
        history.append(w_rob.copy())
    theta = solve(X, y, pen, w0 * w_rob, lb, ub, bounded)
    r = X @ theta - y
    scale = np.full((K, K), np.nan)                     # robust residual scale per batch pair
    for p, q in combinations(range(K), 2):
        m = (ba == p) & (bb == q) & (w0 > 0) & (w_rob > 0)
        scale[p, q] = scale[q, p] = robust_sigma(r[m]) if m.any() else np.nan
    Lq = np.tril(np.ones((Q, Q)))
    drift = np.stack([Lq @ theta[k * Q:(k + 1) * Q] for k in range(K)])
    return dict(knots=knots, drift=drift, weights=w0 * w_rob, scale=scale, history=history)

In [ ]:
def corrupt(tb, frac, seed=5):
    """Replace a fraction of anchors with wrong matches 15–35 bins off; return new tb and their indices."""
    r = np.random.default_rng(seed); n = round(frac * len(tb))
    bad = r.choice(len(tb), n, replace=False); out = tb.copy()
    out[bad] += r.choice([-1, 1], n) * r.uniform(15, 35, n)
    return out, bad

def drift_err(fit, ks=(1, 2, 3)):
    """Max |fitted − true drift| over 35–170 bins (reference gauge, so drift = map to A minus t)."""
    tg = np.linspace(35, 170, 200)
    return max(np.abs(np.interp(tg, fit['knots'], fit['drift'][k]) - (true_f(k, tg) - tg)).max() for k in ks)

tb_bad, bad = corrupt(tb, 0.15)
plain = toy_fit(ba, bb, ta, tb_bad, K, 1.0, REF, Q, n_huber=0, n_tukey=0)
robust = toy_fit(ba, bb, ta, tb_bad, K, 1.0, REF, Q)
print(f'15 % wrong anchors: plain LS max error {drift_err(plain):.2f} bins, robust {drift_err(robust):.2f} bins')
print('wrong anchors with final weight 0:', np.mean(robust['weights'][bad] == 0).round(3))

In [ ]:
H = np.array(robust['history'])                    # (7 rounds, anchors)
good = np.setdiff1d(np.arange(len(ta)), bad)
plt.figure(figsize=(8, 3))
plt.plot(range(1, 8), H[:, good[:60]], color='tab:blue', alpha=0.2)
plt.plot(range(1, 8), H[:, bad[:30]], color='tab:red', alpha=0.4)
plt.axvline(3.5, color='k', ls=':'); plt.text(1.2, 0.5, 'Huber'); plt.text(5, 0.5, 'Tukey')
plt.xlabel('IRLS round'); plt.ylabel('anchor weight'); plt.title('blue: correct anchors, red: wrong anchors')
plt.show()
assert drift_err(robust) < 0.6 and drift_err(plain) > 2 * drift_err(robust)
assert np.mean(robust['weights'][bad] == 0) > 0.95, 'Tukey should drop nearly all wrong anchors'

In [ ]:
def draw_wrong(frac):
    tb_w, _ = corrupt(tb, frac)
    fp = toy_fit(ba, bb, ta, tb_w, K, 1.0, REF, Q, n_huber=0, n_tukey=0)
    fr = toy_fit(ba, bb, ta, tb_w, K, 1.0, REF, Q)
    tg = np.linspace(30, 175, 300); plt.figure(figsize=(8, 3))
    plt.plot(tg, true_f(3, tg) - tg, 'k--', label='true drift of D')
    plt.plot(tg, np.interp(tg, fp['knots'], fp['drift'][3]), label=f'plain LS (err {drift_err(fp):.2f})')
    plt.plot(tg, np.interp(tg, fr['knots'], fr['drift'][3]), label=f'robust IRLS (err {drift_err(fr):.2f})')
    plt.xlabel('RT in batch D (bins)'); plt.ylabel('drift (bins)'); plt.title(f'{frac:.0%} wrong anchors'); plt.legend()
    plt.show()

with_slider(draw_wrong, 'frac', [0.0, 0.05, 0.1, 0.15, 0.25, 0.35, 0.45], 0.15)

**Check your understanding.** Run the robust fit with Huber only (`n_tukey=0`). Do the wrong anchors still get weight 0? What is the error?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
fh = toy_fit(ba, bb, ta, tb_bad, K, 1.0, REF, Q, n_tukey=0)
print("weight-0 fraction", np.mean(fh["weights"][bad] == 0), " error", round(drift_err(fh), 2))   # Huber never gives weight 0

## 12. Maps between batches

**Intuition.** The warps take every batch into the shared clock. To go from batch k to batch l, go into the shared clock with f_k, then come out with the *inverse* of f_l: g_k→l = f_l⁻¹ ∘ f_k. Because every map passes through the same shared clock, going k → m → l gives the same answer as k → l. This is *transitivity*, and it holds by construction.

**Hand example.** f_k(t) = t + 2 and f_l(t) = t − 3. Then f_l⁻¹(τ) = τ + 3, and g_k→l(10) = f_l⁻¹(12) = 15. Via a third batch with f_m(t) = 2t: f_m⁻¹(τ) = τ/2, so g_k→m(10) = 6, and g_m→l(6) = f_l⁻¹(12) = 15. Same answer.

**Inverting.** f_l is monotone, so its inverse exists. The code evaluates f_l on a fine grid of 4·T = 800 points and swaps the axes with `np.interp`. `np.interp` holds the end values outside its range, so **maps clamp at the run ends**.

**Pairwise warps aren't transitive.** If each pair of batches is fitted separately, the three maps around a triplet come from three unrelated fits, and k → m → l need not equal k → l.

| Symbol | Code |
|---|---|
| f_k(t) | `JointWarp.latent(k, t)` |
| f_l⁻¹(τ) | `JointWarp.inverse(l, tau)`, grid `_grid = np.linspace(0, N_BINS − 1, 4 * N_BINS)` |
| g_k→l(t) | `JointWarp.map(k, l, t)` |

In [ ]:
class ToyWarp:
    """The fitted warps as functions (mirrors multibatch.JointWarp)."""
    grid = np.linspace(0, T - 1, 4 * T)
    def __init__(self, fit):
        self.knots, self.drift = fit['knots'], fit['drift']
    def latent(self, k, t):                      # f_k(t) = t + d_k(t)
        return t + np.interp(t, self.knots, self.drift[k])
    def inverse(self, k, tau):                   # f_k⁻¹ by swapping axes on a fine grid
        return np.interp(tau, self.latent(k, self.grid), self.grid)
    def map(self, k, l, t):                      # g_k→l = f_l⁻¹ ∘ f_k
        return self.inverse(l, self.latent(k, np.asarray(t, dtype=float)))

LAM = 1.0
joint = ToyWarp(toy_fit(ba, bb, ta, tb, K, LAM, REF, Q))
tg12 = np.linspace(40, 160, 61)
print('B → C directly vs B → D → C, max difference:',
      np.abs(joint.map(1, 2, tg12) - joint.map(3, 2, joint.map(1, 3, tg12))).max(), 'bins')

`class` bundles data (the knots and drifts) with the functions that use it. `self` is the object itself. The class attribute `grid` is shared by every `ToyWarp`.

In [ ]:
def pairwise(p, q, tb_use=tb):
    """The same fitter on one batch pair's anchors only (like multibatch.fit_pairwise)."""
    m = (ba == p) & (bb == q)
    f = toy_fit(np.zeros(m.sum(), int), np.ones(m.sum(), int), ta[m], tb_use[m], 2, LAM, 0, Q)
    return ToyWarp(f)                            # batch 0 = p, batch 1 = q in this 2-batch fit

pw = {(p, q): pairwise(p, q) for p, q in combinations(range(K), 2)}
def pw_map(k, l, t):
    return pw[(k, l)].map(0, 1, t) if k < l else pw[(l, k)].map(1, 0, t)

cyc_pw = np.abs(pw_map(1, 2, tg12) - pw_map(3, 2, pw_map(1, 3, tg12)))
cyc_j = np.abs(joint.map(1, 2, tg12) - joint.map(3, 2, joint.map(1, 3, tg12)))
plt.plot(tg12, cyc_pw, label='pairwise warps'); plt.plot(tg12, cyc_j, label='joint warps')
plt.xlabel('RT in batch B (bins)'); plt.ylabel('|B→C − B→D→C| (bins)'); plt.legend(); plt.show()

In [ ]:
print(f'triplet B, D, C: pairwise disagree by up to {cyc_pw.max():.3f} bins, joint by {cyc_j.max():.1e} bins')
for k in (1, 2, 3):
    assert np.abs(joint.map(k, 0, tg12) - true_map(k, 0, tg12)).max() < 0.6, 'joint maps recover the truth'
assert cyc_j.max() < 1e-3 and cyc_pw.max() > 10 * cyc_j.max()

The joint maps disagree only by the tiny error of interpolating the inverse on a grid. The pairwise maps disagree by a visible fraction of a bin, even with perfect anchors.

**Check your understanding.** Check that g_k→l and g_l→k undo each other: map times from C to A and back.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
back = joint.map(0, 2, joint.map(2, 0, tg12))
print(np.abs(back - tg12).max())   # ~1e-4: the round trip returns the start

## 13. Borrowing strength

**Intuition.** Batch D has only 2 samples. Suppose it shares only a handful of anchors with the reference A. A *pairwise* fit of D against A can use only those few. The *joint* fit also uses D's anchors with B and C, and B's and C's own anchors with A, so the D → A map is pinned down through the paths D → B → A and D → C → A.

**A caveat.** With *plenty* of good direct anchors, borrowing doesn't help and can cost a little: the joint fit balances all pairs, so it can't follow one pair as closely as a fit dedicated to it. Here, with all 74 direct D–A anchors, the pairwise D → A map is slightly *better* (about 0.4 vs 0.5 bins). Borrowing pays off when direct anchors are few or wrong.

**Hand example.** If D → B is a shift of +3 and B → A is a shift of −9, then D → A is a shift of −6, even with no D–A anchor at all.

In [ ]:
idx_DA = np.flatnonzero((ba == 0) & (bb == 3))          # direct D–A anchors (A is batch 0)

def borrow(n_direct, seed=3):
    """Keep only n_direct D–A anchors. Return max D→A error of the joint and the pairwise fit."""
    keep_DA = np.random.default_rng(seed).choice(idx_DA, n_direct, replace=False)
    keep = np.ones(len(ta), bool); keep[idx_DA] = False; keep[keep_DA] = True
    jf = ToyWarp(toy_fit(ba[keep], bb[keep], ta[keep], tb[keep], K, LAM, REF, Q))
    pf = ToyWarp(toy_fit(np.zeros(n_direct, int), np.ones(n_direct, int), ta[keep_DA], tb[keep_DA], 2, LAM, 0, Q))
    tg = np.linspace(40, 160, 121); true = true_map(3, 0, tg)
    return jf, pf, np.abs(jf.map(3, 0, tg) - true).max(), np.abs(pf.map(1, 0, tg) - true).max()

for n in (2, 3, 5, 10, len(idx_DA)):
    print(f'{n:3d} direct D–A anchors: joint error {borrow(n)[2]:.2f} bins, pairwise {borrow(n)[3]:.2f} bins')

In [ ]:
def draw_borrow(n_direct):
    jf, pf, ej, ep = borrow(n_direct); tg = np.linspace(30, 175, 300); plt.figure(figsize=(8, 3))
    plt.plot(tg, true_map(3, 0, tg) - tg, 'k--', label='true D → A shift')
    plt.plot(tg, jf.map(3, 0, tg) - tg, label=f'joint (err {ej:.2f})')
    plt.plot(tg, pf.map(1, 0, tg) - tg, label=f'pairwise D–A only (err {ep:.2f})')
    plt.xlabel('RT in batch D (bins)'); plt.ylabel('shift into A (bins)')
    plt.title(f'{n_direct} direct D–A anchors'); plt.legend(); plt.show()

with_slider(draw_borrow, 'n_direct', [2, 3, 4, 5, 8, 12, 20, 40, len(idx_DA)], 3)

In [ ]:
e2 = borrow(2)
assert e2[2] < 0.7 and e2[3] > 2 * e2[2], 'with 2 direct anchors the joint fit should be far better'
print('borrowing check OK')

**Check your understanding.** Remove *every* D–A anchor (`n_direct` can't be 0 in `borrow`, so do it by hand). Can the joint fit still map D to A?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
keep = np.ones(len(ta), bool); keep[idx_DA] = False
jf0 = ToyWarp(toy_fit(ba[keep], bb[keep], ta[keep], tb[keep], K, LAM, REF, Q))
tg = np.linspace(40, 160, 121)
print(np.abs(jf0.map(3, 0, tg) - true_map(3, 0, tg)).max())   # still small: the paths through B and C carry it

## 14. Realistic anchors from spectra

So far the anchors came from the true positions. Real anchors must be found in the data. `multibatch.match_anchors(a, b)` does it for one pair of chromatograms, using two building blocks from gcmsalign:

1. **Peaks.** `detect_peaks(tic)`: local maxima of the TIC above its 80th percentile, at least 5 bins apart, at most the 30 tallest (`scipy.signal.find_peaks`).
2. **Fingerprints.** `fingerprint(chroma, pk)`: the mean spectrum of bins pk−1 … pk+1, scaled to length 1.
3. **Cosine similarity.** For unit vectors the dot product is the cosine of the angle between them: 1 for identical spectra, 0 for spectra that share no channel. Hand example: (1, 0, 1)/√2 · (1, 1, 0)/√2 = 1/2.
4. **Drift limit.** Peaks more than 20 min (88.9 bins) apart can't match.
5. **Hungarian matching.** `scipy.optimize.linear_sum_assignment` pairs peaks one-to-one so the total similarity is as large as possible. It *minimises* cost, so the code passes −similarity, with −(−1) for forbidden pairs. Hand example: similarities [[0.9, 0.8], [0.85, 0.1]]. Greedy would take 0.9 first and be left with 0.1 (total 1.0). The optimum is 0.8 + 0.85 = 1.65.
6. **q70 threshold.** Keep a match only if its similarity is at least max(70th percentile of all allowed similarities, 0.30).
7. **Refinement.** For the time in a, fit a parabola through the TIC at the peak and its two neighbours, and take its vertex. For the time in b, search ±3 bins for the best fingerprint match, then fit a parabola through those similarities. Offsets are clipped to ±0.5 bin.
8. **RANSAC.** With ≥ 4 anchors, fit a robust straight line t_b ≈ α·t_a + β (`RANSACRegressor`, residual threshold 3 bins) and drop anchors far from it.
9. **Monotone filter.** Sort by t_a and keep anchors that increase in both t_a and t_b.

In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.signal import find_peaks
from sklearn.linear_model import RANSACRegressor
from gcmsalign.config import MAX_DRIFT_MIN, RANSAC_RESID, REFINE_RADIUS, SIM_THRESHOLD

S = np.array([[0.9, 0.8], [0.85, 0.1]])
rows_, cols_ = linear_sum_assignment(-S)            # maximise similarity = minimise −similarity
print('Hungarian picks', list(zip(rows_, cols_)), 'total', S[rows_, cols_].sum())

def toy_peaks(tic):
    pks, props = find_peaks(tic, height=np.percentile(tic, 80), distance=5)
    if len(pks) > 30:
        pks = pks[np.argsort(props['peak_heights'])[-30:]]
    return np.sort(pks)

def toy_fingerprint(chroma, pk):
    v = chroma[max(0, pk - 1):min(len(chroma), pk + 2)].mean(axis=0)
    n = np.linalg.norm(v)
    return (v / n if n > 1e-8 else v).astype(np.float32)

def parabola_vertex(ym, y0, yp):
    den = ym - 2 * y0 + yp
    return 0.0 if den >= 0 else float(np.clip(0.5 * (ym - yp) / den, -0.5, 0.5))

In [ ]:
def toy_match(a, b):
    """Anchors between chromatograms a and b, step by step as in multibatch.match_anchors."""
    tic_a, tic_b = a.sum(axis=1).astype(float), b.sum(axis=1).astype(float)
    pa, pb = toy_peaks(tic_a), toy_peaks(tic_b)
    if len(pa) < 2 or len(pb) < 2:
        return np.empty((0, 2))
    fa = np.array([toy_fingerprint(a, int(p)) for p in pa]); fb = np.array([toy_fingerprint(b, int(p)) for p in pb])
    sim = fa @ fb.T                                          # cosine similarity, peaks of a × peaks of b
    far = np.abs(pa[:, None] - pb[None, :]) > MAX_DRIFT_MIN / BIN_MIN
    ri, ci = linear_sum_assignment(-np.where(far, -1.0, sim))
    avail = sim[~far]
    thr = max(float(np.percentile(avail, 70)), SIM_THRESHOLD) if avail.size >= 3 else SIM_THRESHOLD
    keep = (sim[ri, ci] >= thr) & ~far[ri, ci]; ri, ci = ri[keep], ci[keep]
    if len(ri) < 2:
        return np.empty((0, 2))
    out = []
    for i, j in zip(ri, ci):
        p = int(pa[i]); t_a = p + (parabola_vertex(*tic_a[p - 1:p + 2]) if 0 < p < T - 1 else 0.0)
        cands = np.arange(max(0, pb[j] - REFINE_RADIUS), min(T, pb[j] + REFINE_RADIUS + 1))
        s = np.array([float(fa[i] @ toy_fingerprint(b, int(c))) for c in cands]); k = int(np.argmax(s))
        out.append((t_a, cands[k] + (parabola_vertex(*s[k - 1:k + 2]) if 0 < k < len(s) - 1 else 0.0)))
    return ransac_and_monotone(np.array(out, dtype=float))

In [ ]:
def ransac_and_monotone(anc):
    if len(anc) >= 4:
        try:
            rs = RANSACRegressor(residual_threshold=RANSAC_RESID, min_samples=2, random_state=0)
            rs.fit(anc[:, :1], anc[:, 1])
            if rs.inlier_mask_.sum() >= 2:
                anc = anc[rs.inlier_mask_]
        except ValueError:
            pass
    anc = anc[np.argsort(anc[:, 0])]
    mono = [0]
    for i in range(1, len(anc)):
        if anc[i, 0] > anc[mono[-1], 0] and anc[i, 1] > anc[mono[-1], 1]:
            mono.append(i)
    return anc[mono] if len(mono) >= 2 else np.empty((0, 2))

m01 = toy_match(chromas[0][0], chromas[1][0])
print(len(m01), 'anchors between A[0] and B[0]:\n', m01.round(2))

Harvested anchors are noisier than the perfect ones. The TIC peak top is shifted slightly by neighbouring compounds, and the fingerprint averages 3 bins. So the recovered warps are a little less accurate, though still within a bin (see the check below).

`anc[:, :1]` keeps the first column as a 2-D table (scikit-learn wants features as a table). `rs.inlier_mask_` is a True/False array marking the anchors that agree with the robust line.

Now harvest every cross-batch sample pair, as `harvest_anchors` does, and compare with the perfect anchors.

In [ ]:
cols = [[] for _ in range(6)]
for p, q in combinations(range(K), 2):
    for i in range(N_SAMPLES[p]):
        for j in range(N_SAMPLES[q]):
            m = toy_match(chromas[p][i], chromas[q][j])
            for col, v in zip(cols, (p, q, i, j)):
                col.append(np.full(len(m), v, dtype=int))
            cols[4].append(m[:, 0]); cols[5].append(m[:, 1])
hba, hbb, hsa, hsb, hta, htb = (np.concatenate(c) for c in cols)
print(len(hta), 'harvested anchors vs', len(ta), 'perfect ones')
fit_h = toy_fit(hba, hbb, hta, htb, K, LAM, REF, Q)
fit_p = toy_fit(ba, bb, ta, tb, K, LAM, REF, Q)
print(f'max drift error: harvested {drift_err(fit_h):.2f} bins, perfect {drift_err(fit_p):.2f} bins')

In [ ]:
true_b = np.array([true_map(p, q, np.array([t]))[0] for p, q, t in zip(hba, hbb, hta)])   # true t_b of each anchor
plt.figure(figsize=(8, 3))
plt.plot(hta, htb - true_b, '.', alpha=0.5)
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('RT in the first batch (bins)'); plt.ylabel('harvested t_b − true t_b (bins)')
plt.title('How far each harvested anchor is from the true correspondence'); plt.show()
print('median |error| of harvested anchors:', np.median(np.abs(htb - true_b)).round(3), 'bins')
assert drift_err(fit_h) < 1.0, 'harvested anchors should still recover the warps to within a bin'
assert np.mean(np.abs(htb - true_b) > 3) < 0.05, 'few harvested anchors should be gross mismatches'

**Check your understanding.** What fraction of harvested anchors are more than 3 bins from the true correspondence (likely mismatches)?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
print(np.mean(np.abs(htb - true_b) > 3).round(3))

## 15. Choosing λ by blocked cross-validation

**Intuition.** *Cross-validation* (CV) tunes a setting by hiding part of the data, fitting on the rest, and scoring the fit on what was hidden. The setting with the best held-out score wins. We never look at the truth (in real data there is none) and never at TIC r.

**Which part to hide matters.** Hiding whole *samples* leaves the held-out anchors at the same compounds' times as the training anchors. A wiggly drift that passes exactly through every compound looks perfect there, so sample folds can't see wiggles *between* compounds. MBA hides whole **windows of retention time**: blocks of 12 bins, with block b going to fold b mod 5. An anchor is held out if *either* of its ends falls in the fold's blocks, and used for training only if *neither* does. The score is the median |error| of predicting t_b from t_a. This measures interpolation across gaps, which is what a wiggle gets wrong.

**Hand example.** t = 30 is in block ⌊30/12⌋ = 2, so fold 2. t = 70 is in block 5, so fold 0. An anchor (30, 70) is held out in folds 2 and 0.

| Symbol | Code (`scripts/30_multibatch_align.py`) |
|---|---|
| block size, folds | `RT_BLOCK = 12`, `N_FOLDS = 5` |
| block → fold | `blk = lambda t: (np.floor(t / RT_BLOCK).astype(int)) % N_FOLDS` |
| held-out error | `anchor_errors(model, anc)` = `model.map(a, b, ta) − tb` |

In [ ]:
blk = lambda t: (np.floor(t / 12).astype(int)) % 5
print('folds of t = 30 and 70:', blk(np.array([30., 70])))

def anchor_errors(warp, ba_, bb_, ta_, tb_):
    err = np.empty(len(ta_))
    for p, q in combinations(range(K), 2):
        m = (ba_ == p) & (bb_ == q)
        if m.any():
            err[m] = warp.map(p, q, ta_[m]) - tb_[m]
    return err

def cv_score(lam, folds, Qc=21):
    errs = []
    for tr, te in folds:
        w = ToyWarp(toy_fit(ba[tr], bb[tr], ta[tr], tb[tr], K, lam, REF, Qc))
        errs.append(anchor_errors(w, ba[te], bb[te], ta[te], tb[te]))
    return float(np.median(np.abs(np.concatenate(errs))))

In [ ]:
blocked = [((blk(ta) != f) & (blk(tb) != f), (blk(ta) == f) | (blk(tb) == f)) for f in range(5)]
r15 = np.random.default_rng(0)
sfold = [r15.permutation(n) % 5 for n in N_SAMPLES]           # each sample → a fold, within its batch
fa_ = np.array([sfold[k][s] for k, s in zip(ba, sa)]); fb_ = np.array([sfold[k][s] for k, s in zip(bb, sb)])
by_sample = [((fa_ != f) & (fb_ != f), (fa_ == f) | (fb_ == f)) for f in range(5)]
LAMBDAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
cv_blk = [cv_score(l, blocked) for l in LAMBDAS]
cv_smp = [cv_score(l, by_sample) for l in LAMBDAS]
plt.semilogx(LAMBDAS, cv_blk, 'o-', label='RT-blocked folds (MBA)'); plt.semilogx(LAMBDAS, cv_smp, 's-', label='sample folds')
plt.xlabel('λ'); plt.ylabel('median held-out |error| (bins)'); plt.legend(); plt.show()

In [ ]:
lam_cv = LAMBDAS[int(np.argmin(cv_blk))]
print('blocked CV:', np.round(cv_blk, 3), '→ chosen λ =', lam_cv)
print('sample CV: ', np.round(cv_smp, 3))
gap_blk, gap_smp = cv_blk[0] - min(cv_blk), cv_smp[0] - min(cv_smp)
print(f'penalty for the smallest λ: blocked {gap_blk:.3f} bins, sample folds {gap_smp:.3f} bins')
assert lam_cv not in (LAMBDAS[0], LAMBDAS[-1]), 'blocked CV should pick an interior λ'
assert gap_blk > 3 * gap_smp, 'blocked folds should punish an under-smoothed fit much more than sample folds'

Sample folds barely notice that the smallest λ is too wiggly. Blocked folds punish it clearly. The real script chooses λ from the same kind of curve, over `LAMBDAS = [0.1, 1, 10, …, 10000]`, with the reference gauge.

**Check your understanding.** Which fold is the anchor (t_a, t_b) = (13, 100) held out in?

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
print(sorted(set(blk(np.array([13., 100])))))   # block 1 → fold 1; block 8 → fold 3

## 16. Bootstrap uncertainty

**Intuition.** How much would the warps change with a different set of samples? The *bootstrap* imitates drawing new data by resampling what we have: within each batch, draw samples *with replacement*, refit, repeat. The spread of the refitted warps estimates the uncertainty.

**Multiplicities as weights.** Instead of copying samples, the code gives each anchor a weight. A batch with 3 samples might draw counts (2, 0, 1): sample 0 twice, sample 1 never, sample 2 once (`rng.multinomial`). An anchor between sample i of batch k and sample j of batch l gets weight count_k[i] × count_l[j]. With counts (2, 0, 1) and (1, 1), the anchor (sample 0, sample 1) gets 2 × 1 = 2.

**Band and predictive interval.**
- The **band** is the 5th–95th percentile of the refitted maps at each time. It describes uncertainty in the *warp*.
- The **predictive interval** is for a *new anchor*, which also has its own scatter around the warp. Its half-width is √(half-band² + (1.645·σ_kl)²), where σ_kl is the pair's robust residual scale and 1.645 is the 90 % point of a normal distribution.

**Coverage.** A 90 % interval should contain the truth 90 % of the time. We know the truth here, so we can check.

In [ ]:
def boot_weights(rng):
    counts = [rng.multinomial(n, np.full(n, 1.0 / n)) for n in N_SAMPLES]     # per batch
    return np.array([counts[p][i] * counts[q][j] for p, q, i, j in zip(ba, bb, sa, sb)], dtype=float)

r16 = np.random.default_rng(11)
print('one draw of counts for a 3-sample batch:', np.random.default_rng(1).multinomial(3, [1 / 3] * 3), '(sample 1 twice, sample 2 never)')
tg16 = np.linspace(40, 160, 61)
boots = [ToyWarp(toy_fit(ba, bb, ta, tb, K, LAM, REF, Q, w0=boot_weights(r16))) for _ in range(100)]
band = {k: np.percentile(np.stack([b.map(k, 0, tg16) for b in boots]), [5, 95], axis=0) for k in (1, 2, 3)}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, k in zip(axes, (1, 2, 3)):                  # everything relative to the truth, so the band is visible
    tr = true_map(k, 0, tg16)
    ax.fill_between(tg16, band[k][0] - tr, band[k][1] - tr, alpha=0.4, label='90 % bootstrap band')
    ax.plot(tg16, joint.map(k, 0, tg16) - tr, label='joint fit')
    ax.axhline(0, color='k', ls='--', label='truth')
    ax.set(xlabel=f'RT in batch {NAMES[k]} (bins)', title=f'{NAMES[k]} → A')
axes[0].set_ylabel('map − true map (bins)'); axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
cov = {NAMES[k]: np.mean((true_map(k, 0, tg16) >= band[k][0]) & (true_map(k, 0, tg16) <= band[k][1])) for k in band}
print('share of times where the band contains the truth:', {k: round(v, 2) for k, v in cov.items()}, '(target 0.90)')

**Under-coverage, and why.** The bands (where the shaded area covers the dashed zero line) contain the truth far less than 90 % of the time. A bootstrap measures *variance*: how much the fit moves when the data changes. It can't see *bias*, a systematic error that every refit shares. The smoothness penalty creates bias, pulling every fit towards straighter drifts than the truth, and each refit is pulled the same way. The band is centred in the wrong place, and too narrow to reach the truth. The real fish-oil intervals under-cover for the same reason (81 % instead of 90 % for the predictive interval). The paper suggests calibration, such as conformal scaling, as the fix.

In [ ]:
fit16 = toy_fit(ba, bb, ta, tb, K, LAM, REF, Q)
k = 3; m = (ba == 0) & (bb == k)                       # anchors between A and D
pred = joint.map(0, k, ta[m])
q = np.percentile(np.stack([b.map(0, k, ta[m]) for b in boots]), [5, 95], axis=0)
hw_band = (q[1] - q[0]) / 2
hw_pi = np.sqrt(hw_band ** 2 + (1.645 * fit16['scale'][0, k]) ** 2)
inside = np.abs(tb[m] - pred) <= hw_pi
print(f'σ_AD = {fit16["scale"][0, k]:.3f} bins; predictive intervals contain {inside.mean():.0%} of A–D anchors')
assert max(cov.values()) < 0.9, 'the warp-only band under-covers the truth (bias, not variance)'
assert inside.mean() > max(cov.values()), 'adding anchor scatter widens the interval and raises coverage'

**Check your understanding.** Repeat the band at λ = 0.01 (less smoothing, so less bias). Does coverage of the truth go up? Use 40 refits.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
rb = np.random.default_rng(12)
bs = [ToyWarp(toy_fit(ba, bb, ta, tb, K, 0.01, REF, Q, w0=boot_weights(rb))) for _ in range(40)]
for k in (1, 2, 3):
    qq = np.percentile(np.stack([b.map(k, 0, tg16) for b in bs]), [5, 95], axis=0); tr = true_map(k, 0, tg16)
    print(NAMES[k], np.mean((tr >= qq[0]) & (tr <= qq[1])).round(2))
# Coverage rises close to 90 %: with less smoothing there is less bias, so the bands reach the truth.

## 17. Metrics

**TIC correlation.** After warping batch B into A's frame, how similar are their TICs? The *Pearson correlation* r measures how well two curves rise and fall together: 1 for the same shape, 0 for unrelated shapes. It is computed only from 5 min onwards (bin 22, `EVAL_START_BIN = int(5.0 // 0.225)`), because the real runs started acquiring at different times, and shared zeros at the start would inflate r.

**Hand example.** x = (1, 2, 3) and y = (2, 4, 7). Centred: (−1, 0, 1) and (−2.33, −0.33, 2.67). r = Σxy / √(Σx²·Σy²) = 5 / √(2 · 12.67) = 0.993.

**Held-out anchor error.** The median |error| of predicting held-out anchors, the score from Section 15. It measures correspondence directly. TIC r rewards overlapping peaks even when the wrong compounds overlap.

| Symbol | Code |
|---|---|
| TIC r | `gcmsalign.metrics.pearson_rows(a, b)`, on `EVAL_BINS = slice(22, 200)` |

In [ ]:
def pearson(x, y, start=int(5.0 // BIN_MIN)):
    x, y = np.asarray(x, float)[start:], np.asarray(y, float)[start:]
    x, y = x - x.mean(), y - y.mean()
    return float(x @ y / np.sqrt((x @ x) * (y @ y)))

print('hand example r =', round(pearson([1, 2, 3], [2, 4, 7], start=0), 3))
src = joint.map(0, 1, np.arange(T, dtype=float))          # for each bin of A, where to read B
r_un = np.mean([pearson(a, b) for a in tics[0] for b in tics[1]])
r_al = np.mean([pearson(a, np.interp(src, np.arange(T), b)) for a in tics[0] for b in tics[1]])
print(f'TIC r, A vs B: unaligned {r_un:.3f}, after the joint warp {r_al:.3f}')
assert r_al > r_un + 0.2 and abs(pearson([1, 2, 3], [2, 4, 7], start=0) - 0.993) < 1e-3

In [ ]:
heldout = np.concatenate([anchor_errors(ToyWarp(toy_fit(ba[tr], bb[tr], ta[tr], tb[tr], K, lam_cv, REF, 21)),
                                        ba[te], bb[te], ta[te], tb[te]) for tr, te in blocked])
plt.hist(np.abs(heldout), bins=40); plt.xlabel('|held-out anchor error| (bins)'); plt.ylabel('anchors')
plt.title(f'RT-blocked held-out error at λ = {lam_cv}: median {np.median(np.abs(heldout)):.3f} bins'); plt.show()

**Check your understanding.** Compute TIC r between A and C before and after warping C into A's frame.

*The solution is in the next cell, which is collapsed: click it to expand.*

In [ ]:
# Solution
srcC = joint.map(0, 2, np.arange(T, dtype=float))
print(np.mean([pearson(a, c) for a in tics[0] for c in tics[2]]).round(3), np.mean([pearson(a, np.interp(srcC, np.arange(T), c)) for a in tics[0] for c in tics[2]]).round(3))

## 18. Bridge to the real code

Everything above was written from scratch. Now we run the repository's real implementation on **the same toy anchors** and check that the answers agree. The real code is `src/multibatch.py` (the model), with helpers from `scripts/30_multibatch_align.py` (bootstrap weights, anchor errors) and `gcmsalign.metrics` (TIC r).

The real code uses an `Anchors` object with the same six columns we used (`ba, bb, sa, sb, ta, tb`).

In [ ]:
import multibatch as mb
from analysis import load_script                  # small helper that imports a numbered script
from gcmsalign.metrics import pearson_rows

s30 = load_script('30_multibatch_align.py', 's30_tutorial')
anc_real = mb.Anchors(ba, bb, sa, sb, ta, tb)
cases = [dict(lam=1.0, gauge=REF), dict(lam=0.01, gauge=REF), dict(lam=100.0, gauge='mean')]
for cs in cases:
    toy = toy_fit(ba, bb, ta, tb, K, cs['lam'], cs['gauge'], Q)
    real = mb.fit_joint(anc_real, K, lam=cs['lam'], gauge=cs['gauge'], n_knots=Q)
    print(f"λ={cs['lam']:<6} gauge={cs['gauge']!s:<5} drift {np.abs(toy['drift'] - real.drift).max():.1e}  "
          f"weights {np.abs(toy['weights'] - real.weights).max():.1e}  "
          f"scale {np.nanmax(np.abs(toy['scale'] - real.scale)):.1e}  (bins)")
    assert np.abs(toy['drift'] - real.drift).max() < 1e-6 and np.abs(toy['weights'] - real.weights).max() < 1e-6

In [ ]:
# The robust case (wrong anchors) and a bootstrap-weighted case.
tb_bad2, _ = corrupt(tb, 0.15)
w_b = boot_weights(np.random.default_rng(3))
for label, tbx, w in (('15 % wrong anchors', tb_bad2, None), ('bootstrap weights', tb, w_b)):
    toy = toy_fit(ba, bb, ta, tbx, K, 1.0, REF, Q, w0=w)
    real = mb.fit_joint(mb.Anchors(ba, bb, sa, sb, ta, tbx), K, lam=1.0, gauge=REF, base_weight=w, n_knots=Q)
    print(f'{label:20s} drift difference {np.abs(toy["drift"] - real.drift).max():.1e} bins')
    assert np.abs(toy['drift'] - real.drift).max() < 1e-6

real1 = mb.fit_joint(anc_real, K, lam=1.0, gauge=REF, n_knots=Q)
print('maps (JointWarp.map vs ToyWarp.map):', np.abs(real1.map(3, 0, tg12) - joint.map(3, 0, tg12)).max().round(12))

In [ ]:
# Anchor harvesting: our toy_match against multibatch.match_anchors, every cross-batch sample pair.
worst, n_pairs = 0.0, 0
for p, q in combinations(range(K), 2):
    for i in range(N_SAMPLES[p]):
        for j in range(N_SAMPLES[q]):
            mine, theirs = toy_match(chromas[p][i], chromas[q][j]), mb.match_anchors(chromas[p][i], chromas[q][j])
            assert mine.shape == theirs.shape, (p, q, i, j)
            worst = max(worst, np.abs(mine - theirs).max() if len(mine) else 0.0); n_pairs += 1
print(f'toy_match == match_anchors on all {n_pairs} sample pairs (max difference {worst:.1e} bins)')

# Bootstrap weights, held-out errors and TIC r against the script / gcmsalign versions.
assert np.array_equal(boot_weights(np.random.default_rng(9)), s30.boot_weights(anc_real, N_SAMPLES, np.random.default_rng(9)))
assert np.allclose(anchor_errors(joint, ba, bb, ta, tb), s30.anchor_errors(real1, anc_real), atol=1e-9)
assert np.isclose(pearson(tics[0][0], tics[1][0]), pearson_rows(tics[0][0], tics[1][0])[0])
print('boot_weights, anchor_errors and TIC r agree with the real code')

**Side by side.** What each piece you wrote corresponds to in the real code (see `CODE_MAP.md` for more):

| You wrote (section) | Real code | Agreement checked above |
|---|---|---|
| `hat_basis` (3) | `multibatch._hat_basis` | through the fit |
| `L = np.tril(np.ones((Q, Q)))` (4) | `Lcum` in `fit_joint` | through the fit |
| `design_matrix` (7) | the `X`, `y` loop in `fit_joint` | through the fit |
| `gauge_rows` (8) | `G_rows`, weight `GAUGE_WEIGHT * n_eff / M` | ref and mean gauge |
| `smooth_rows` (9) | `D2`, `lam_eff`, block-diagonal `P` | λ = 0.01, 1, 100 |
| `build_problem`, `solve` (10) | `P`, `lb`, `ub`, Cholesky + `lsq_linear(..., 'bvls')` | stacked rows ≡ Cholesky route |
| `robust_sigma`, IRLS in `toy_fit` (11) | `_robust_sigma`, the Huber/Tukey loop | weights, wrong anchors |
| `ToyWarp` (12) | `JointWarp.latent / inverse / map` | maps |
| `pairwise` (12) | `fit_pairwise` | same fitter, one pair |
| `toy_match` (14) | `match_anchors` (+ gcmsalign `detect_peaks`, `fingerprint`) | every sample pair |
| blocked CV (15) | the `blk` folds in script 30 | same rule |
| `boot_weights` (16) | `boot_weights` in script 30 | identical draws |
| `anchor_errors` (15) | `anchor_errors` in script 30 | identical |
| `pearson` (17) | `gcmsalign.metrics.pearson_rows` | identical |

**Where the code differs from the paper's summary** (all followed in this notebook):
1. T = 200 bins is fixed (`N_BINS` in gcmsalign); knots and the inverse grid always span 0 … 199.
2. A ridge of 10⁻⁸·N_eff·I is added to the penalty. It isn't in the paper, and it is too small to matter.
3. Both gauges are *penalties* with weight 10³·N_eff/Q, not exact constraints. The reference drift is pinned only approximately (to 10⁻⁵–10⁻⁴ bins here).
4. The solve uses the normal equations, a Cholesky factor and BVLS rather than stacked rows. This is mathematically equivalent.
5. σ̂ uses only anchors with positive bootstrap weight. The per-pair scale σ_kl uses only anchors that survive the final Tukey round.
6. IRLS runs 7 reweighting rounds, then one final solve (8 solves).
7. The hat basis clamps times outside the knot range, and maps clamp at the run ends.
8. Only a common *constant* leaves the residuals exactly unchanged. Other common changes to the shared clock alter the fit slightly, so the two gauges give maps about 0.2 bins apart here.
9. The predictive interval's half-width is half the band width, centred on the point estimate (not on the band's centre).

## Appendix: the neural refinement, in brief

`src/neural_joint.py` (`NeuralJointWarp`, `train`) is an optional step on top of the joint model. Instead of one warp per *batch*, it gives every *sample* i its own monotone map g_i from the shared clock to its own clock. g_i starts exactly at the joint model and then learns small corrections:

- the corrections are log-slope adjustments from two small neural networks (one per batch, one per sample), squashed with tanh so a slope never changes by more than a factor of 2, and integrated, so every g_i stays monotone;
- the reference batch's batch-level correction is held at zero (the gauge);
- it trains on the joint model's anchors (with their robust weights, so rejected anchors stay rejected) plus a *dense* term that compares each sample's whole m/z-resolved chromatogram with the average of the other batches in the shared frame.

In the fair benchmark it is best on drifted pairs under the midpoint protocol and slightly behind the joint model in batch B's frame, so the paper treats it as a variant, not the main method.

## Glossary

| Term | Meaning |
|---|---|
| Anchor | A pair of times (a, b): the same compound, matched by its spectrum, seen at a in one batch and at b in another |
| Augmented least squares | Writing a penalty as extra rows under the design matrix with target 0 |
| Batch | Samples measured together; drift differs between batches |
| Bias | A systematic error that every refit shares; a bootstrap can't see it |
| Bootstrap | Resampling the data with replacement to estimate how much a fit varies |
| BVLS | Bounded-variable least squares, the solver used with box bounds |
| Chromatogram | Signal by retention time × m/z channel for one run |
| Cosine similarity | Dot product of two unit-length vectors; 1 = same direction |
| Coverage | How often an interval contains the truth; a 90 % interval should do so 90 % of the time |
| Cross-validation | Hiding part of the data, fitting on the rest, scoring on the hidden part |
| Design matrix (X) | One row per equation (anchor), one column per parameter |
| Drift d_k | How far batch k's clock is from the shared clock at each time |
| Gauge | The freedom to shift the shared clock; fixed by the reference or mean gauge |
| Hat basis | Linear-interpolation weights; each knot's weight is a triangle ("hat") |
| Huber / Tukey | Robust weight functions: Huber shrinks large residuals' weights, Tukey zeroes them |
| Hungarian algorithm | Optimal one-to-one matching (`linear_sum_assignment`) |
| IRLS | Iteratively reweighted least squares |
| Knot | A time where the piecewise-linear drift is pinned; straight lines in between |
| MAD | Median absolute deviation, a scale that outliers can't inflate |
| Monotone | Never decreasing; a monotone warp never reverses the order of compounds |
| Normal equations | XᵀXθ = Xᵀy, solved by the least-squares θ |
| Null space | Directions θ that X ignores (Xθ = 0); they make XᵀX singular |
| Pearson r | Correlation of two curves' shapes, between −1 and 1 |
| RANSAC | Robust line fitting: fit to random small subsets and keep the line most points agree with |
| Regularisation | A penalty that prefers simpler (here, straighter) fits |
| Retention time (RT) | When a compound leaves the column; measured in bins (0.225 min each) |
| Singular matrix | A matrix with determinant 0, which can't be inverted |
| TIC | Total ion chromatogram: the chromatogram summed over m/z |
| Transitive | k → m → l gives the same map as k → l |
| Warp f_k | A monotone function from batch k's clock to the shared clock |